# 03 — Model, objective, and training mechanics

What this notebook is for: showing that the **model and the objective do what
they claim**, before any results are reported. Nothing here is a result. It is
the evidence that the machinery underneath Notebook 04 is sound.

Four things are demonstrated:

1. The architecture, and the parameter cost of each backbone.
2. The class-balanced weights — that they actually favour the rare classes.
3. The hierarchical consistency term — that it is *doing something*, rather than
   sitting inert at zero.
4. A deliberate overfitting test on a tiny subset. A model that cannot drive the
   loss to ~0 on 200 images has a bug, and no amount of training on 29,000 will
   fix it.

All code lives in `src/`. This notebook imports it, so what is demonstrated here
is the same code the experiments run — not a re-implementation that can drift.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch

from src import cache, config, engine, experiments, losses, metrics, splits, viz
from src.models import HierarchicalClassifier, BACKBONES, count_parameters
from src.hierarchy import CLASSES, FINE_NAMES

viz.apply_style()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")
if DEVICE == "cuda":
    print(f"gpu:    {torch.cuda.get_device_name(0)}")

## 1. Architecture

One shared pretrained backbone feeding two linear heads. `mode="flat"` never
constructs the lineage head at all — an unused head would still appear in the
parameter count and in weight decay, which would quietly make the "identical
backbone" comparison less identical than it claims.

In [ ]:
rows = []
for key, timm_name in BACKBONES.items():
    m = HierarchicalClassifier(key, mode="hier", pretrained=False)
    p = count_parameters(m)
    rows.append({
        "backbone": key,
        "timm name": timm_name,
        "feature dim": m.head_fine.in_features,
        "parameters (M)": round(p["total"] / 1e6, 1),
    })

pd.DataFrame(rows)

Note the feature dimension is *measured*, not read from
`backbone.num_features`. For MobileNetV3 that attribute reports 960 — the width
before `conv_head` — while the real pooled output is 1280. Trusting it builds
heads of the wrong width and fails at the first forward pass.

In [ ]:
# Both modes, one class. Shapes are what the loss and metrics assume downstream.
x = torch.randn(4, 3, config.IMAGE_SIZE, config.IMAGE_SIZE)

for mode in ("hier", "flat"):
    m = HierarchicalClassifier("resnet", mode=mode, pretrained=False).eval()
    out = m(x)
    y1_hat, y2_hat = m.predict(x)
    lin = tuple(out.logits1.shape) if out.logits1 is not None else None
    print(f"mode={mode:5s}  logits2={tuple(out.logits2.shape)}  logits1={lin}  "
          f"predict -> y1{tuple(y1_hat.shape)} y2{tuple(y2_hat.shape)}")

In flat mode there is no lineage head, so `predict` *derives* the
lineage from the fine prediction through the fixed class→lineage map. That is
not a convenience: it is what makes the within- vs cross-lineage error analysis
computable for the baseline too, and therefore what makes the arms comparable on
the analysis the dissertation turns on.

## 2. Class-balanced weights

Cui et al.'s effective-number reweighting. The claim to check is simply that the
weights track rarity, and that they are normalised to mean 1 — without that
normalisation the weighted arm would train at a different effective learning
rate, and the imbalance ablation would confound reweighting with step size.

In [ ]:
split_df = splits.load()
train_counts = (split_df[split_df.split == "train"]["y2"]
                .value_counts().reindex(range(18), fill_value=0))
counts_t = torch.tensor(train_counts.to_numpy(), dtype=torch.float32)

w = losses.class_balanced_weights(counts_t)

tbl = pd.DataFrame({
    "class": list(FINE_NAMES),
    "train count": train_counts.to_numpy(),
    "CB weight": w.numpy().round(3),
    "minority": [c.idx in metrics.MINORITY_IDX for c in CLASSES],
}).sort_values("train count")

print(f"weights mean = {w.mean():.4f}   (normalised to 1.0)")
print(f"max/min ratio = {w.max()/w.min():.1f}   vs raw count ratio "
      f"{train_counts.max()/train_counts.min():.1f}")
tbl

The weight ratio is far smaller than the raw count ratio. That is
the point of the effective-number formulation: samples of one class overlap in
feature space, so the *n*-th sample adds less information than the first, and
pure inverse-frequency weighting over-corrects.

## 3. Is the hierarchy term doing anything?

The consistency term marginalises the fine posterior into lineage space and
penalises disagreement with the lineage head. Two sanity checks:

- On a **hierarchically consistent** prediction it should be ~0.
- On a **contradictory** one (fine head says lymphoid, lineage head says
  myeloid) it should be clearly positive.

If it cannot distinguish those, the term is inert and the model is only
multi-task, not hierarchical.

In [ ]:
crit = losses.HierarchicalLoss(counts_t, use_hierarchy=True, use_imbalance=True)

# Class 0 is lymphoid (lineage 0); class 7 (myeloblast) is myeloid (lineage 1).
def one_hot_logits(idx, n, scale=10.0):
    z = torch.full((1, n), -scale)
    z[0, idx] = scale
    return z

agree    = crit.consistency(one_hot_logits(0, 3), one_hot_logits(0, 18))
disagree = crit.consistency(one_hot_logits(1, 3), one_hot_logits(0, 18))
uniform  = crit.consistency(torch.zeros(1, 3),    torch.zeros(1, 18))

print(f"heads agree      (lymphoid / typical lymphocyte) : {float(agree):.4f}")
print(f"heads contradict (myeloid  / typical lymphocyte) : {float(disagree):.4f}")
print(f"both uninformative (uniform)                     : {float(uniform):.4f}")

### Ablation switches

Each arm is the same objective with different field values, so the switches must
visibly change which loss terms exist.

In [ ]:
m = HierarchicalClassifier("resnet", mode="hier", pretrained=False)
out = m(torch.randn(8, 3, 224, 224))
y1 = torch.randint(0, 3, (8,))
y2 = torch.randint(0, 18, (8,))

for name, use_h, use_i in [
    ("full hierarchical", True,  True),
    ("- imbalance",       True,  False),
    ("- hierarchy",       False, True),
]:
    c = losses.HierarchicalLoss(counts_t, use_hierarchy=use_h, use_imbalance=use_i)
    loss, parts = c(out, y1, y2)
    print(f"{name:20s} total={float(loss):7.4f}  terms present: {sorted(parts.keys() - {'total'})}")

## 4. Overfitting test

The strongest cheap check available. A correctly wired model must be able to
memorise a tiny subset: if the loss will not fall to near zero on 200 images,
something is broken in the data path, the labels, or the gradient flow, and
training on 29,000 images will only hide it.

This trains for a few dozen steps and takes well under a minute.

In [ ]:
from torch.utils.data import DataLoader
from src.dataset import MLL23Dataset

cache_path = cache.validate_cache(cache.RAW_CACHE_PATH, split_df)
tiny = split_df[split_df.split == "train"].sample(200, random_state=0)

# train=False: no augmentation. Memorisation is the thing being tested, and
# augmentation would make the target move.
ds = MLL23Dataset(tiny, train=False, cache_path=cache_path)
dl = DataLoader(ds, batch_size=32, shuffle=True, num_workers=0)

model = HierarchicalClassifier("resnet", mode="hier", pretrained=True).to(DEVICE)
crit  = losses.HierarchicalLoss(counts_t.to(DEVICE)).to(DEVICE)
opt   = torch.optim.AdamW(model.parameters(), lr=1e-4)

model.train()
trace = []
for epoch in range(30):
    tot, n = 0.0, 0
    for xb, y1b, y2b in dl:
        xb, y1b, y2b = xb.to(DEVICE), y1b.to(DEVICE), y2b.to(DEVICE)
        loss, _ = crit(model(xb), y1b, y2b)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        tot += float(loss); n += 1
    trace.append(tot / n)
    if epoch % 5 == 0 or epoch == 29:
        print(f"  epoch {epoch:2d}  loss {trace[-1]:.5f}")

print(f"\nfirst {trace[0]:.4f} -> last {trace[-1]:.5f}"
      f"   ({'PASS - model can memorise' if trace[-1] < trace[0] * 0.1 else 'FAIL - investigate'})")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(trace, color=viz.LINEAGE_COLORS["Lymphoid"], lw=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("Training loss")
ax.set_title("Overfitting test: 200 images, no augmentation", pad=10)
ax.set_yscale("log")
fig.tight_layout()
fig.savefig(config.ARTIFACT_DIR / "overfit_check.png")
plt.show()

## 5. The experiment matrix

The arms as they will be run. Note that experiments 1 and 4 from the proposal are
the **same configuration** — "the hierarchical model with the hierarchy removed"
*is* the flat single-head baseline. It is trained once and reported twice; the
write-up must say so rather than implying five independent runs.

In [ ]:
experiments.arm_table()

In [ ]:
cfgs = experiments.main_configs()
print(f"Phase 2: {len(cfgs)} runs = {len(experiments.ARMS)} distinct configs "
      f"x {len(experiments.SEEDS)} seeds")
for c in cfgs[:4]:
    print(f"  {c.run_id}")
print("  ...")

---

## Summary

- The two-head architecture builds and runs on all four backbones, with the
  feature width measured rather than assumed.
- Class-balanced weights favour rare classes and are normalised to mean 1, so the
  imbalance ablation is not confounded with effective learning rate.
- The consistency term separates agreeing from contradicting heads, so the
  hierarchy is a mechanism rather than a label.
- The model memorises a 200-image subset, so the data path and gradient flow are
  sound.

Notebook 04 runs the experiments.